In [1]:
import os.path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

In [2]:
data = pd.read_csv("../datas/text_classify/train_tokens.csv", header=None, sep="\t")
data

,0,1
0,还有 双鸭山 到 淮阴 的 汽车票 吗 13号 的,Travel-Query
1,从 这里 怎么 回家,Travel-Query
2,随便 播放 一 首 专辑 阁楼 里 的 佛 里 的 歌,Music-Play
3,给 看 一下 墓 王 之 王 嘛,FilmTele-Play
4,我 想 看 挑战 两 把 s686 打 突变 团竞 的 游戏 视频,Video-Play
...,...,...
12095,一千六百五十三 加 三千一百六十五点六五 等于 几,Calendar-Query
12096,稍 小 点 客厅 空调 风速,HomeAppliance-Control
12097,黎耀祥 陈豪 邓萃雯 畲诗曼 陈法拉 敖嘉年 杨怡 马浚伟 等 到场 出席,Radio-Listen
12098,百事 盖世 群星 星光 演唱会 有 谁,Video-Play


In [3]:
y = data.iloc[:,1]
encoder = LabelEncoder()
y_labeled = encoder.fit_transform(y)
print(encoder.classes_)

['Alarm-Update' 'Audio-Play' 'Calendar-Query' 'FilmTele-Play'
 'HomeAppliance-Control' 'Music-Play' 'Other' 'Radio-Listen'
 'TVProgram-Play' 'Travel-Query' 'Video-Play' 'Weather-Query']


In [4]:
vocab = pd.read_csv('../datas/text_classify/vocab.csv', sep='\t', header=None)
vocab.head()
X_id = pd.read_csv('../datas/text_classify/corpus_id.csv', sep='\t', header=None)
X_id.head()

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,9315,2776,2391,6689,7440,6462,2983,89,7440,0,...,0,0,0,0,0,0,0,0,0,0
1,1634,9342,4851,3286,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9946,5378,805,10231,1130,9796,9588,7440,1801,9588,...,0,0,0,0,0,0,0,0,0,0
3,8085,7516,806,3432,7124,1279,7124,3260,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5011,4946,7516,5248,1170,5127,775,5072,7775,3313,...,0,0,0,0,0,0,0,0,0,0


In [5]:
ture_lens = np.count_nonzero(X_id, axis=1).tolist()
ture_lens

[9,
 4,
 12,
 8,
 13,
 12,
 10,
 11,
 8,
 10,
 8,
 10,
 8,
 11,
 8,
 10,
 4,
 4,
 10,
 7,
 7,
 5,
 5,
 14,
 7,
 11,
 13,
 15,
 8,
 13,
 9,
 6,
 6,
 16,
 9,
 10,
 10,
 5,
 9,
 5,
 17,
 9,
 8,
 8,
 5,
 9,
 5,
 12,
 5,
 4,
 8,
 15,
 10,
 9,
 6,
 12,
 13,
 13,
 7,
 12,
 8,
 9,
 10,
 10,
 9,
 14,
 7,
 9,
 7,
 10,
 10,
 5,
 5,
 7,
 13,
 9,
 9,
 11,
 9,
 6,
 6,
 10,
 9,
 9,
 8,
 5,
 7,
 10,
 9,
 6,
 12,
 7,
 3,
 7,
 13,
 9,
 7,
 6,
 6,
 6,
 10,
 4,
 10,
 11,
 3,
 9,
 5,
 11,
 7,
 6,
 10,
 7,
 4,
 13,
 18,
 6,
 5,
 10,
 7,
 6,
 7,
 9,
 11,
 10,
 12,
 12,
 12,
 10,
 8,
 6,
 4,
 15,
 10,
 7,
 11,
 15,
 9,
 7,
 10,
 7,
 12,
 13,
 3,
 9,
 11,
 13,
 7,
 7,
 6,
 15,
 8,
 9,
 8,
 11,
 9,
 12,
 8,
 7,
 13,
 8,
 8,
 13,
 15,
 10,
 11,
 15,
 10,
 6,
 12,
 10,
 9,
 7,
 11,
 9,
 11,
 18,
 9,
 8,
 9,
 11,
 16,
 9,
 11,
 11,
 8,
 6,
 10,
 6,
 6,
 9,
 10,
 10,
 7,
 6,
 9,
 5,
 7,
 5,
 5,
 9,
 7,
 6,
 10,
 9,
 7,
 10,
 12,
 7,
 5,
 9,
 4,
 10,
 10,
 11,
 8,
 8,
 7,
 7,
 14,
 6,
 7,
 18,
 8,
 10,
 9,
 11,
 9,


In [6]:
X_train,X_test,y_train,y_test,ture_lens_train,ture_lens_test = train_test_split(X_id,y_labeled,ture_lens,test_size=0.2,random_state=24)

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
from 意图识别训练.npl_model import Network_RNN_Bidirection, Network_LSTM,Network_GRU_Bidirection, MyDataset

In [8]:
vocab_size = len(vocab)
num_classes = len(encoder.classes_)
embed_dim = 128
hidden_size = 256
batch_size = 64
learning_rate = 0.01
epochs = 40
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [9]:
# RNN
train_loader = DataLoader(MyDataset(X_train,y_train,ture_lens_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(MyDataset(X_test,y_test,ture_lens_test), batch_size=batch_size//2, shuffle=False)

model = Network_RNN_Bidirection(vocab_size,embed_dim,hidden_size,num_classes).to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss().to(device)

In [10]:
# LSTM
model = Network_LSTM(vocab_size,embed_dim,hidden_size,num_classes).to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss().to(device)

In [14]:
# GRU
model = Network_GRU_Bidirection(vocab_size,embed_dim,hidden_size,num_classes).to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss().to(device)

In [18]:
summary_dir = "../datas/text_classify/summary/GRU_B"
best_model_path = "../datas/text_classify/model/GRU_B_best_model.pkl"
last_model_path = "../datas/text_classify/model/GRU_B_last_model.pkl"

writer = SummaryWriter(log_dir=summary_dir)
start_epoch = 0
best_acc = 0.0
resume_training = True

if os.path.exists(last_model_path) and resume_training:
    print(f"发现历史保存模型 '{last_model_path}'，正在加载...")
    checkpoint = torch.load(last_model_path,map_location=device,weights_only=False)

    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])

    start_epoch = checkpoint["epoch"] + 1
    best_acc = checkpoint['best_acc']
    print(f"成功恢复训练，将从 Epoch {start_epoch} 开始，历史最佳准确率: {best_acc:.2f}%\n")

for epoch in range(start_epoch,start_epoch+epochs):
    model.train()
    train_loss_sum = 0.0

    train_pbar = tqdm(train_loader, desc=f'Train Epoch {epoch}/{start_epoch+epochs-1}')
    for batch_idx, (data, target, true_len) in enumerate(train_pbar):
        data,target,true_len = data.to(device), target.to(device),true_len.cpu()

        optimizer.zero_grad()
        output = model(data,true_len)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        global_step = epoch * len(train_loader) + batch_idx
        writer.add_scalar('Train/Step_Loss', loss.item(), global_step)

    avg_train_loss = train_loss_sum / len(train_loader)
    writer.add_scalar('Train/Avg_Loss', avg_train_loss, epoch)

    model.eval()
    test_loss = 0.0
    correct = 0

    test_pbar = tqdm(test_loader, desc=f'Test Epoch  {epoch}/{start_epoch+epochs-1}', leave=False)
    with torch.no_grad():
        for data, target, true_len in test_pbar:
            data, target, true_len = data.to(device), target.to(device), true_len.cpu()
            output = model(data, true_len)

            # 累加 batch_loss
            batch_loss = criterion(output, target).item()
            test_loss += batch_loss

            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            test_pbar.set_postfix({'loss': f'{batch_loss:.4f}'})

    # 计算平均测试损失和准确率（修正了原代码中的逻辑）
    avg_test_loss = test_loss / len(test_loader)
    accuracy = 100. * correct / len(test_loader.dataset)

    # 将测试指标记录到 TensorBoard
    writer.add_scalar('Test/Epoch_Loss', avg_test_loss, epoch)
    writer.add_scalar('Test/Accuracy', accuracy, epoch)

    print(f'Epoch {epoch} 完成 | Test Avg Loss: {avg_test_loss:.4f} | Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)')

    checkpoint = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_acc': best_acc
    }
    if epoch == start_epoch+epochs-1:
        print("保存本轮最后一个模型")
        torch.save(checkpoint, last_model_path)

    if accuracy > best_acc:
        print(f'>>> 发现新的最佳准确率: {accuracy:.2f}% (前最佳: {best_acc:.2f}%)，正在保存最佳模型...\n')
        best_acc = accuracy
        checkpoint['best_acc'] = best_acc  # 更新 checkpoint 里的最佳记录
        torch.save(checkpoint, best_model_path)
    else:
        print() # 输出空行以便于阅读

writer.close()
print("训练全部完成！")


发现历史保存模型 'datas/text_classify/model/GRU_B_last_model.pkl'，正在加载...
成功恢复训练，将从 Epoch 120 开始，历史最佳准确率: 86.03%



Train Epoch 120/159:   0%|          | 0/152 [00:00<?, ?it/s]

Test Epoch  120/159:   0%|          | 0/76 [00:00<?, ?it/s]

Epoch 120 完成 | Test Avg Loss: 0.5168 | Accuracy: 2081/2420 (85.99%)



Train Epoch 121/159:   0%|          | 0/152 [00:00<?, ?it/s]

...

Test Epoch  158/159:   0%|          | 0/76 [00:00<?, ?it/s]

Epoch 158 完成 | Test Avg Loss: 0.5087 | Accuracy: 2092/2420 (86.45%)



Train Epoch 159/159:   0%|          | 0/152 [00:00<?, ?it/s]

Test Epoch  159/159:   0%|          | 0/76 [00:00<?, ?it/s]

Epoch 159 完成 | Test Avg Loss: 0.5124 | Accuracy: 2095/2420 (86.57%)
保存本轮最后一个模型

训练全部完成！


In [25]:
model.train()
for epoch in range(epochs):
    for batch_idx, (data, target,true_len) in enumerate(train_loader):
        data,target,true_len = data.to(device), target.to(device),true_len.cpu()

        optimizer.zero_grad()
        output = model(data,true_len)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')


Train Epoch: 0 [0/9680] Loss: 2.488906
Train Epoch: 0 [6400/9680] Loss: 2.330427
Train Epoch: 1 [0/9680] Loss: 2.244824
Train Epoch: 1 [6400/9680] Loss: 2.187381
Train Epoch: 2 [0/9680] Loss: 2.085022
Train Epoch: 2 [6400/9680] Loss: 1.904608
Train Epoch: 3 [0/9680] Loss: 1.952807
Train Epoch: 3 [6400/9680] Loss: 1.939640
Train Epoch: 4 [0/9680] Loss: 1.688882
Train Epoch: 4 [6400/9680] Loss: 1.765607
Train Epoch: 5 [0/9680] Loss: 1.827712
Train Epoch: 5 [6400/9680] Loss: 1.631160
Train Epoch: 6 [0/9680] Loss: 1.502374
Train Epoch: 6 [6400/9680] Loss: 1.672210
Train Epoch: 7 [0/9680] Loss: 1.608522
Train Epoch: 7 [6400/9680] Loss: 1.444830
Train Epoch: 8 [0/9680] Loss: 1.345589
Train Epoch: 8 [6400/9680] Loss: 1.401740
Train Epoch: 9 [0/9680] Loss: 1.335472
Train Epoch: 9 [6400/9680] Loss: 1.288782
Train Epoch: 10 [0/9680] Loss: 1.175314
Train Epoch: 10 [6400/9680] Loss: 1.346113
Train Epoch: 11 [0/9680] Loss: 1.210952
Train Epoch: 11 [6400/9680] Loss: 1.292554
Train Epoch: 12 [0/9680]

In [26]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target,true_len in test_loader:
        data, target,true_len = data.to(device), target.to(device),true_len.cpu()
        output = model(data,true_len)
        test_loss += criterion(output, target).item()
        pred = output.argmax(dim=1, keepdim=True)
        # output.sort(dim=1, descending=True)

        correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')


Test set: Average loss: 0.0149, Accuracy: 2091/2420 (86.40%)

